In [1]:
!pip install torch transformers langchain-huggingface langgraph grandalf langgraph-checkpoint-sqlite


In [2]:
# langgraph_persistent_chat.py

import torch
import langchain
import gc
import re
import sqlite3
from typing import TypedDict, Annotated, List, Literal
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

# IMPORT CHECKPOINTER
# This is the key component for crash recovery
try:
    from langgraph.checkpoint.sqlite import SqliteSaver
except ImportError:
    print("Please install the checkpoint library: pip install langgraph-checkpoint-sqlite")
    # Fallback for demonstration if package is missing (won't survive process kill)
    from langgraph.checkpoint.memory import MemorySaver as SqliteSaver

# =============================================================================
# CONFIGURATION
# =============================================================================
for attr in ["verbose", "debug"]:
    if not hasattr(langchain, attr): setattr(langchain, attr, False)
if not hasattr(langchain, "llm_cache"): langchain.llm_cache = None

def clear_gpu_memory():
    """Clears GPU memory to prevent OOM errors."""
    try:
        for var in ['llama_pipe', 'qwen_pipe', 'llama_lc', 'qwen_lc']:
            if var in globals(): del globals()[var]
    except: pass
    gc.collect()
    torch.cuda.empty_cache()

def get_device():
    if torch.cuda.is_available(): return "cuda"
    elif torch.backends.mps.is_available(): return "mps"
    return "cpu"

# =============================================================================
# STATE DEFINITION
# =============================================================================
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    should_exit: bool
    # Note: 'verbose' is not stored in DB to allow changing it on restart

# =============================================================================
# MODEL LOADING
# =============================================================================
def load_model(model_id, device):
    print(f"Loading: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if "llama" in model_id.lower():
        tokenizer.pad_token_id = tokenizer.eos_token_id
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16 if device != "cpu" else torch.float32,
        device_map=device if device == "cuda" else None,
    )
    if device == "mps": model = model.to(device)

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        truncation=True,
        return_full_text=False,
        max_length=None
    )
    return HuggingFacePipeline(pipeline=pipe), pipe

def create_models():
    device = get_device()
    l_lc, l_pipe = load_model("meta-llama/Llama-3.2-1B-Instruct", device)
    q_lc, q_pipe = load_model("Qwen/Qwen2.5-1.5B-Instruct", device)
    return (l_lc, l_pipe), (q_lc, q_pipe)

# =============================================================================
# PROMPT LOGIC
# =============================================================================
def build_prompt(messages: List[BaseMessage], target_model: Literal["Llama", "Qwen"]) -> str:
    if target_model == "Llama":
        system_text = "You are Llama. Discuss with a Human and AI Qwen. Do NOT start lines with 'Llama:'."
        prompt = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n" + system_text + "<|eot_id|>"
    else:
        system_text = "You are Qwen. Discuss with a Human and AI Llama. Do NOT start lines with 'Qwen:'."
        prompt = "<|im_start|>system\n" + system_text + "<|im_end|>\n"

    for msg in messages:
        role = ""
        content = msg.content

        if isinstance(msg, HumanMessage):
            role = "user"
            content = f"Human: {content}"
        elif isinstance(msg, AIMessage):
            author = msg.name if hasattr(msg, 'name') and msg.name else "AI"
            if author == target_model:
                role = "assistant"
            else:
                role = "user"
                content = f"{author}: {content}"

        if target_model == "Llama":
            prompt += f"<|start_header_id|>{role}<|end_header_id|>\n\n{content}<|eot_id|>"
        else:
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"

    if target_model == "Llama": prompt += "<|start_header_id|>assistant<|end_header_id|>\n\n"
    else: prompt += "<|im_start|>assistant\n"

    return prompt

def clean_response(text: str) -> str:
    return re.sub(r"^(Llama|Qwen|AI|Assistant|Human):\s*", "", text, flags=re.IGNORECASE).strip()

# =============================================================================
# GRAPH NODES
# =============================================================================
def create_graph(llama_comps, qwen_comps, checkpointer):
    llama_lc, llama_raw = llama_comps
    qwen_lc, qwen_raw = qwen_comps

    def get_user_input(state: AgentState) -> dict:
        # Check history to see if we are resuming
        history_len = len(state.get("messages", []))
        if history_len > 0:
            print(f"\n[SYSTEM] Resuming conversation with {history_len} previous messages.")
            # Print last message context
            last_msg = state["messages"][-1]
            print(f"[LAST MESSAGE] {last_msg.name if hasattr(last_msg, 'name') else 'User'}: {last_msg.content[:50]}...")

        print("\n" + "="*60)
        print("Enter text (Start with 'Hey Qwen' to switch):")
        print("="*60)
        try:
            raw = input("\n> ").strip()
        except EOFError: return {"should_exit": True}

        if raw.lower() in ['quit', 'exit', 'q']: return {"should_exit": True}

        return {"messages": [HumanMessage(content=raw)], "should_exit": False}

    def call_llama(state: AgentState) -> dict:
        prompt = build_prompt(state["messages"], target_model="Llama")
        try:
            response = llama_lc.invoke(prompt)
        except:
            out = llama_raw(prompt)
            txt = out[0]["generated_text"]
            response = txt[len(prompt):] if txt.startswith(prompt) else txt
        return {"messages": [AIMessage(content=clean_response(response), name="Llama")]}

    def call_qwen(state: AgentState) -> dict:
        prompt = build_prompt(state["messages"], target_model="Qwen")
        try:
            response = qwen_lc.invoke(prompt)
        except:
            out = qwen_raw(prompt)
            txt = out[0]["generated_text"]
            response = txt[len(prompt):] if txt.startswith(prompt) else txt
        return {"messages": [AIMessage(content=clean_response(response), name="Qwen")]}

    def print_response(state: AgentState) -> dict:
        last_msg = state["messages"][-1]
        name = last_msg.name if hasattr(last_msg, 'name') else "AI"
        print("\n" + "-"*60)
        print(f"🗣️  {name} Says:")
        print("-"*60)
        print(last_msg.content)
        return {}

    def route_after_input(state: AgentState):
        if state.get("should_exit"): return END
        if not state["messages"]: return "get_user_input"

        last_input = state["messages"][-1].content.lower()
        if last_input.startswith("hey qwen"): return "call_qwen"
        else: return "call_llama"

    # --- BUILD GRAPH ---
    workflow = StateGraph(AgentState)
    workflow.add_node("get_user_input", get_user_input)
    workflow.add_node("call_llama", call_llama)
    workflow.add_node("call_qwen", call_qwen)
    workflow.add_node("print_response", print_response)

    workflow.add_edge(START, "get_user_input")
    workflow.add_conditional_edges(
        "get_user_input",
        route_after_input,
        {END: END, "get_user_input": "get_user_input", "call_llama": "call_llama", "call_qwen": "call_qwen"}
    )
    workflow.add_edge("call_llama", "print_response")
    workflow.add_edge("call_qwen", "print_response")
    workflow.add_edge("print_response", "get_user_input")

    # COMPILE WITH CHECKPOINTER
    return workflow.compile(checkpointer=checkpointer)

def main():
    clear_gpu_memory()

    # 1. SETUP PERSISTENCE
    # We use a context manager to ensure the DB connection closes properly
    # This creates a file 'checkpoints.sqlite' in your current directory
    with SqliteSaver.from_conn_string("checkpoints.sqlite") as checkpointer:

        print("Initializing Persistent Chat...")
        llama_comps, qwen_comps = create_models()

        # 2. CREATE GRAPH WITH CHECKPOINTER
        graph = create_graph(llama_comps, qwen_comps, checkpointer)

        # 3. DEFINE THREAD ID
        # This is what identifies the session. If you change this, you get a fresh chat.
        # Ideally, you'd ask the user for an ID or generate one.
        thread_id = "conversation_1"
        config = {"configurable": {"thread_id": thread_id}}

        print(f"\n[INFO] Running with Thread ID: {thread_id}")
        print("[INFO] You can kill this script anytime. Run it again to resume!")

        # 4. INVOKE
        # We pass an empty state update.
        # - If it's a new thread, it starts fresh.
        # - If it's an existing thread, it loads the state from SQLite and ignores the empty messages.
        initial_state = {"messages": [], "should_exit": False}

        graph.invoke(initial_state, config=config)

if __name__ == "__main__":
    main()

Initializing Persistent Chat...
Loading: meta-llama/Llama-3.2-1B-Instruct...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'max_new_tokens', 'top_p', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Loading: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


[INFO] Running with Thread ID: conversation_1
[INFO] You can kill this script anytime. Run it again to resume!

[SYSTEM] Resuming conversation with 2 previous messages.
[LAST MESSAGE] Llama: Ah, okay! My secret codeword is "Giraffe"...

Enter text (Start with 'Hey Qwen' to switch):

> What was the codeword?

------------------------------------------------------------
🗣️  Llama Says:
------------------------------------------------------------
The codeword was "Giraffe".

[SYSTEM] Resuming conversation with 4 previous messages.
[LAST MESSAGE] Llama: The codeword was "Giraffe"....

Enter text (Start with 'Hey Qwen' to switch):


KeyboardInterrupt: Interrupted by user